In [13]:
import sys
from pathlib import Path
import yaml

import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [11]:
emb_folder = project_root / "data" / "mvp" / "ProtLM_embeddings_with_labels_layer8"
test_organisms = ["Koxy", "psRCH2", "Korea", "BFirm"]
val_organisms = ["WCS417", "Putida", "DdiaME23", "Burk376"]
train_organisms = [
    "ANA3",
    "Cup4G11",
    "Dda3937",
    "Ddia6719",
    "Dyella79",
    "HerbieS",
    "Keio",
    "MR1",
    "PV4",
    "Pedo557",
    "SB2B",
    "Smeli",
    "acidovorax_3H11",
    "azobra",
    "pseudo13_GW456_L13",
    "pseudo1_N1B4",
    "pseudo3_N2E3",
    "pseudo5_N2C3_1",
    "pseudo6_N2E2"
]


In [33]:
test = "/home/ds85/projects/GeneEssentiality/data/mvp/ProtLM_embeddings_with_labels_layer8/acidovorax_3H11_proteomelm.pt"
test_1 = test.split("/")[-1].split("_proteomelm.pt")[0]
print(test_1)

data = torch.load(test, map_location="cpu", weights_only=False)
print(data.keys())
print(data["embeddings"].shape)
print(data["y"].shape)


acidovorax_3H11
dict_keys(['embeddings', 'group_labels', 'y'])
torch.Size([4964, 1152])
torch.Size([4964])


In [53]:
# Use lists to collect tensors
train_emb, train_y = [], []
val_emb, val_y = [], []
test_emb, test_y = [], []

for file in emb_folder.glob("*.pt"):
    org_id = file.stem.split("_proteomelm")[0]
    data = torch.load(file, map_location="cpu", weights_only=False)
    if not isinstance(data, dict) or "embeddings" not in data or "y" not in data:
        continue
    emb = data["embeddings"].float()
    y = data["y"] if isinstance(data["y"], torch.Tensor) else torch.tensor(data["y"], dtype=torch.long)
    if org_id in train_organisms:
        train_emb.append(emb)
        train_y.append(y)
    elif org_id in val_organisms:
        val_emb.append(emb)
        val_y.append(y)
    elif org_id in test_organisms:
        test_emb.append(emb)
        test_y.append(y)

# Concatenate into single tensors per split
X_train = torch.cat(train_emb)
y_train = torch.cat(train_y, dim=0)
X_val = torch.cat(val_emb)
y_val = torch.cat(val_y, dim=0)
X_test = torch.cat(test_emb)
y_test = torch.cat(test_y, dim=0)

print("Before dropping no_data:")
print(f"Train X: {X_train.shape}, Train y: {y_train.shape}")
print(f"Val X: {X_val.shape}, Val y: {y_val.shape}")
print(f"Test X: {X_test.shape}, Test y: {y_test.shape}")
# Drop no_data (y == -1). 
# 0 = essential (always_essential+conditional)
# 1 = non_essential 
mask_train = y_train >= 0
X_train, y_train = X_train[mask_train], y_train[mask_train].clone()
y_train[y_train == 1] = 0
y_train[y_train == 2] = 1
mask_val = y_val >= 0
X_val, y_val = X_val[mask_val], y_val[mask_val].clone()
y_val[y_val == 1] = 0
y_val[y_val == 2] = 1
mask_test = y_test >= 0
X_test, y_test = X_test[mask_test], y_test[mask_test].clone()
y_test[y_test == 1] = 0
y_test[y_test == 2] = 1

print()
print("After dropping no_data:")
print(f"Train X: {X_train.shape}, Train y: {y_train.shape}")
print(f"Val X: {X_val.shape}, Val y: {y_val.shape}")
print(f"Test X: {X_test.shape}, Test y: {y_test.shape}")


Before dropping no_data:
Train X: torch.Size([95807, 1152]), Train y: torch.Size([95807])
Val X: torch.Size([21806, 1152]), Val y: torch.Size([21806])
Test X: torch.Size([20905, 1152]), Test y: torch.Size([20905])

After dropping no_data:
Train X: torch.Size([78663, 1152]), Train y: torch.Size([78663])
Val X: torch.Size([17620, 1152]), Val y: torch.Size([17620])
Test X: torch.Size([16577, 1152]), Test y: torch.Size([16577])


In [ ]:
# Minimal MLP: one hidden layer. No dropout, no class weights, no project code.
input_dim = 1152
hidden_dim = 2048
n_classes = 2
batch_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



model = nn.Sequential(
    nn.Linear(input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim, n_classes),
).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [57]:
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size)

In [58]:
# Training loop with validation each epoch (train vs val loss for plotting).

train_loss = []
val_loss = []

epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    mean_train = total_loss / n_batches
    train_loss.append(mean_train)
    # Validation (no gradients)
    model.eval()
    val_total = 0.0
    val_batches = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            val_total += loss_fn(logits, y).item()
            val_batches += 1
    mean_val = val_total / val_batches if val_batches > 0 else 0.0
    val_loss.append(mean_val)
    print(f"Epoch {epoch}: train_loss = {mean_train:.4f}  val_loss = {mean_val:.4f}")

Epoch 0: train_loss = 0.4088  val_loss = 0.3597


KeyboardInterrupt: 

In [ ]:
# Plot train vs val loss
import matplotlib.pyplot as plt
plt.plot(train_loss, label="Train")
plt.plot(val_loss, label="Val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()